# Generar tripletas y embeddings 

In [2]:
# -*- coding: utf-8 -*-
import os
import sys
import random
import json
import pandas as pd
import numpy as np
import torch
from types import SimpleNamespace

import sys
# Apuntar a la carpeta raíz donde está model.py
sys.path.append(r"C:\Users\56946\TuckER") 
from model import TuckER

# ============================================================
# 🔹 CONFIGURACIÓN GLOBAL
# ============================================================

# ⚠️ CARPETA SALIDA (4D)
OUTPUT_DATASET_DIR = r"../../data/dataset_2019_2020_2021_binario_4d"
OUTPUT_EMBEDDINGS_DIR = r"../../notebooks/Experimento_warm_start/embeddings_4d_binario"

DF_BASE = r"../../mis_scripts/dataframes_por_semestre"

# Pares de semestres (Cohortes)
PARES_SEMESTRES = [
    ("20191", "20192"), 
    ("20201", "20202"), 
    ("20211", "20212")
]

# Definición de Cursos
CURSOS_PRIMER  = {"MA1101", "MA1001", "FI1000", "BT1211"}
CURSOS_SEGUNDO = {"MA1002", "MA1102", "FI1100", "CC1002"}
CURSOS_PERMITIDOS = CURSOS_PRIMER.union(CURSOS_SEGUNDO)

# Configuración Modelo
EDIM = 4  # ⚠️ SOLO 4 NOTAS
RDIM_DUMMY = 2 
SPLIT_RATIO = [0.8, 0.1, 0.1]
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# 🔹 FASE 1: GENERACIÓN DE TRIPLETAS (Lógica Estricta)
# ============================================================

def normalizar_columnas(df):
    df["ID"] = df["ID"].astype(str).str.strip().str.upper()
    df["CURSO"] = df["CURSO"].astype(str).str.strip().str.upper()
    df["ESTADO_CURSO"] = df["ESTADO_CURSO"].astype(str)
    return df

def determinar_relacion(estado, nota_val):
    estado = str(estado)
    if "Aprobado" in estado: return "aprueba"
    if "Reprobado" in estado or "Eliminado" in estado: return "reprueba"
    try:
        val = float(str(nota_val).replace(",", "."))
        return "reprueba" if val < 4.0 else "aprueba"
    except:
        return None

def generar_tripletas():
    print("\n--- 🔨 FASE 1: GENERANDO TRIPLETAS BINARIAS (3 COHORTES) ---")
    os.makedirs(OUTPUT_DATASET_DIR, exist_ok=True)
    
    tripletas = []
    ids_validos_global = set() 

    for sem_ant, sem_act in PARES_SEMESTRES:
        ruta_prev = os.path.join(DF_BASE, f"df_{sem_ant}.csv")
        ruta_act  = os.path.join(DF_BASE, f"df_{sem_act}.csv")
        
        if not os.path.exists(ruta_prev) or not os.path.exists(ruta_act):
            print(f"⚠️ Saltando {sem_ant}->{sem_act}")
            continue

        print(f"   Procesando: {sem_ant} -> {sem_act}")
        df_prev = normalizar_columnas(pd.read_csv(ruta_prev, sep=";"))
        df_act  = normalizar_columnas(pd.read_csv(ruta_act,  sep=";"))

        # 1. Filtro S1: Toman los 4 cursos fundamentales
        df_fund_prev = df_prev[df_prev["CURSO"].isin(CURSOS_PRIMER)]
        conteo = df_fund_prev.groupby("ID")["CURSO"].nunique()
        alumnos_cumplen_s1 = set(conteo[conteo == len(CURSOS_PRIMER)].index)
        
        # 2. Filtro S2: Toman AL MENOS 1 de los 8 cursos fundamentales
        df_fund_act = df_act[df_act["CURSO"].isin(CURSOS_PERMITIDOS)]
        alumnos_cumplen_s2 = set(df_fund_act["ID"].unique())
        
        # 3. INTERSECCIÓN
        alumnos_validos = alumnos_cumplen_s1.intersection(alumnos_cumplen_s2)
        
        print(f"     -> Cumplen S1 (4 ramos): {len(alumnos_cumplen_s1)}")
        print(f"     -> Cumplen S2 (>=1 ramo): {len(alumnos_cumplen_s2)}")
        print(f"     -> INTERSECCIÓN VÁLIDA: {len(alumnos_validos)}")
        
        ids_validos_global.update(alumnos_validos)

        # 4. Generar Tripletas
        df_filtrado = df_act[
            (df_act["ID"].isin(alumnos_validos)) &
            (df_act["CURSO"].isin(CURSOS_PERMITIDOS))
        ]

        count_local = 0
        for _, row in df_filtrado.iterrows():
            rel = determinar_relacion(row["ESTADO_CURSO"], row["NOTA"])
            if rel:
                tripletas.append(f"{row['ID']}\t{rel}\t{row['CURSO']}")
                count_local += 1
        
        print(f"     -> Tripletas generadas: {count_local}")

    # Shuffle y Split
    random.shuffle(tripletas)
    n = len(tripletas)
    n_train = int(n * SPLIT_RATIO[0])
    n_valid = int(n * SPLIT_RATIO[1])
    
    train_data = tripletas[:n_train]
    valid_data = tripletas[n_train : n_train + n_valid]
    test_data  = tripletas[n_train + n_valid :]
    
    print(f"\n📊 Total Tripletas: {n}")
    
    def guardar(nombre, data):
        with open(os.path.join(OUTPUT_DATASET_DIR, nombre), 'w', encoding='utf-8') as f:
            f.write("\n".join(data) + "\n")
            
    guardar("train.txt", train_data)
    guardar("valid.txt", valid_data)
    guardar("test.txt", test_data)
    
    # Balanceo
    reprobados = [t for t in train_data if "\treprueba\t" in t]
    aprobados = [t for t in train_data if "\taprueba\t" in t]
    
    if len(reprobados) > 0 and len(aprobados) > len(reprobados):
        factor = len(aprobados) // len(reprobados)
        resto = len(aprobados) % len(reprobados)
        train_bal = aprobados + (reprobados * factor) + reprobados[:resto]
        random.shuffle(train_bal)
        guardar("train_balanceado.txt", train_bal)
        print(f"   ⚖️ Balanceo: {len(reprobados)} orig -> {len(train_bal) - len(aprobados)} final")
    else:
        guardar("train_balanceado.txt", train_data)

    print(f"✅ Dataset guardado en: {OUTPUT_DATASET_DIR}")
    return ids_validos_global

# ============================================================
# 🔹 FASE 2: GENERACIÓN DE EMBEDDINGS 4D (SOLO NOTAS)
# ============================================================

def normalizar_nota(n):
    if pd.isna(n): return -1.0
    return (n - 4.0) / 3.0

def construir_matriz_notas(rutas_s1):
    df_list = []
    for r in rutas_s1:
        if os.path.exists(r):
            df_list.append(pd.read_csv(r, sep=";"))
    if not df_list: return pd.DataFrame()
            
    df_total = pd.concat(df_list, ignore_index=True)
    df_total = normalizar_columnas(df_total)
    
    # Filtrar solo fundamentales S1
    df_total = df_total[df_total["CURSO"].isin(CURSOS_PRIMER)]
    
    df_total['NOTA'] = pd.to_numeric(df_total['NOTA'], errors='coerce')
    pivot = df_total.pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
    return pivot.reindex(columns=list(CURSOS_PRIMER))

def get_vocab_generated():
    entities = set()
    relations = set()
    for fname in ["train.txt", "valid.txt", "test.txt"]:
        path = os.path.join(OUTPUT_DATASET_DIR, fname)
        if not os.path.exists(path): continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                h, r, t = line.strip().split()
                entities.add(h); entities.add(t); relations.add(r)
    
    d = SimpleNamespace()
    d.entities = sorted(list(entities))
    d.relations = sorted(list(relations)) + [r + "_reverse" for r in sorted(list(relations))]
    d.entity_idxs = {e: i for i, e in enumerate(d.entities)}
    d.relation_idxs = {r: i for i, r in enumerate(d.relations)}
    return d

def generar_embeddings(ids_validos):
    print("\n--- 🧠 FASE 2: INICIALIZANDO EMBEDDINGS 4D (Solo Notas) ---")
    os.makedirs(OUTPUT_EMBEDDINGS_DIR, exist_ok=True)
    
    d = get_vocab_generated()
    print(f"   Vocabulario: {len(d.entities)} entidades.")
    
    # Cargar notas S1 de TODOS los años
    rutas_s1 = [
        os.path.join(DF_BASE, "df_20191.csv"),
        os.path.join(DF_BASE, "df_20201.csv"),
        os.path.join(DF_BASE, "df_20211.csv")
    ]
    tabla_notas = construir_matriz_notas(rutas_s1)
    
    # Inicializar TuckER con EDIM=4
    modelo = TuckER(d, EDIM, RDIM_DUMMY, input_dropout=0, hidden_dropout1=0, hidden_dropout2=0)
    
    count = 0
    with torch.no_grad():
        for entity, idx in d.entity_idxs.items():
            if entity in ids_validos and entity in tabla_notas.index:
                row = tabla_notas.loc[entity]
                
                # ⚠️ Solo 4 Notas Normalizadas [-1, 1]
                vec_notas = [normalizar_nota(row[c]) for c in CURSOS_PRIMER]
                
                final_vec = np.array(vec_notas, dtype=np.float32)
                
                if final_vec.shape[0] != 4:
                    print(f"⚠️ Error dimensión: {entity} tiene {final_vec.shape}")
                    continue

                modelo.E.weight[idx] = torch.tensor(final_vec)
                count += 1

    print(f"   ✅ Se inicializaron {count} alumnos con datos reales 4D.")
    
    path_pt = os.path.join(OUTPUT_EMBEDDINGS_DIR, "embeddings_inicializados_binarios_4d.pt")
    path_json = os.path.join(OUTPUT_EMBEDDINGS_DIR, "vocabulario_binario_4d.json")
    
    torch.save(modelo.E.weight.data, path_pt)
    with open(path_json, 'w', encoding='utf-8') as f:
        json.dump({"entities": d.entities, "relations": d.relations}, f, indent=2)
        
    print(f"💾 Guardado: {path_pt}")
    print(f"💾 Guardado: {path_json}")

if __name__ == "__main__":
    ids_val = generar_tripletas()
    generar_embeddings(ids_val)


--- 🔨 FASE 1: GENERANDO TRIPLETAS BINARIAS (3 COHORTES) ---
   Procesando: 20191 -> 20192
     -> Cumplen S1 (4 ramos): 832
     -> Cumplen S2 (>=1 ramo): 802
     -> INTERSECCIÓN VÁLIDA: 790
     -> Tripletas generadas: 2961
   Procesando: 20201 -> 20202
     -> Cumplen S1 (4 ramos): 811
     -> Cumplen S2 (>=1 ramo): 877
     -> INTERSECCIÓN VÁLIDA: 787
     -> Tripletas generadas: 3021
   Procesando: 20211 -> 20212
     -> Cumplen S1 (4 ramos): 830
     -> Cumplen S2 (>=1 ramo): 869
     -> INTERSECCIÓN VÁLIDA: 783
     -> Tripletas generadas: 3023

📊 Total Tripletas: 9005
   ⚖️ Balanceo: 398 orig -> 6806 final
✅ Dataset guardado en: ../../data/dataset_2019_2020_2021_binario_4d

--- 🧠 FASE 2: INICIALIZANDO EMBEDDINGS 4D (Solo Notas) ---
   Vocabulario: 2368 entidades.
   ✅ Se inicializaron 2353 alumnos con datos reales 4D.
💾 Guardado: ../../notebooks/Experimento_warm_start/embeddings_4d_binario\embeddings_inicializados_binarios_4d.pt
💾 Guardado: ../../notebooks/Experimento_warm_sta

### Entrenamiento de las redes

In [5]:
# -*- coding: utf-8 -*-
import os, sys, json
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.optim as optim
from sklearn.model_selection import train_test_split
from types import SimpleNamespace

# =========================
# CONFIGURACIÓN GENERAL
# =========================
DEVICE = torch.device("cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# -------------------------------------------------
# 1. Rutas del Modelo TuckER (3 COHORTES)
# -------------------------------------------------
# Carpeta donde están los train/valid/test txt de las 3 cohortes
TUCKER_DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_2021_binario_4d"

# Carpeta donde están los resultados del entrenamiento TuckER 3 cohortes
RESULTS_BASE    = r"C:\Users\56946\TuckER\results"

# Prefijo de tus carpetas de resultados (Ajusta si le pusiste otro nombre en el .bat)
# Ejemplo: Experimento_warm_start_rdim6_1000epochs_earlystopping_2019_2020_2021_patience300_balanceado_4d
RUN_PREFIX      = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_2021_patience400"

# -------------------------------------------------
# 2. Rutas de Datos y Salida
# -------------------------------------------------
BASE_DF_PATH    = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"

# Carpeta donde se guardarán las redes neuronales entrenadas
SAVE_BASE       = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_3cohortes_modelobalanceado"

# Cursos Fundamentales
CURSOS_PRIMER  = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PERMITIDOS = CURSOS_PRIMER + CURSOS_SEGUNDO

RDIMS = list(range(1, 17))  # 1..16

# =========================
# FUNCIONES AUXILIARES
# =========================
def cargar_df_notas(path_csv):
    """Lee un df de notas, normaliza ID y CURSO."""
    if not os.path.exists(path_csv):
        print(f"⚠️ Warning: No existe {path_csv}")
        return pd.DataFrame(columns=['ID', 'CURSO', 'NOTA'])
        
    df = pd.read_csv(path_csv, sep=';')
    df['ID']    = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    return df

def get_vocab_from_data_dir(data_dir):
    """Reconstruye vocabulario de entidades y relaciones."""
    entities, relations = set(), set()
    for part in ['train.txt', 'valid.txt', 'test.txt']:
        path = os.path.join(data_dir, part)
        if not os.path.exists(path): continue
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                h, r, t = line.strip().split()
                entities.add(h); entities.add(t); relations.add(r)
    relations = sorted(list(relations))
    relations_with_reverse = relations + [r + '_reverse' for r in relations]
    return sorted(list(entities)), sorted(list(set(relations_with_reverse)))

def pick_state_dict(ckpt_loaded):
    if isinstance(ckpt_loaded, dict):
        if "model_state_dict" in ckpt_loaded: return ckpt_loaded["model_state_dict"]
        if "state_dict" in ckpt_loaded: return ckpt_loaded["state_dict"]
    return ckpt_loaded

def cargar_E_weights(path_pt):
    sd = pick_state_dict(torch.load(path_pt, map_location=DEVICE))
    if "E.weight" not in sd: raise KeyError(f"E.weight missing in {path_pt}")
    return sd["E.weight"].detach().cpu()

def construir_notas_generacion(df_sem1, df_sem2, cursos_primer, cursos_permitidos):
    """Construye vector de notas S1 para alumnos que siguen en S2."""
    if df_sem1.empty or df_sem2.empty: return pd.DataFrame()

    # 1. Alumnos con 4 cursos en S1
    df_fund = df_sem1[df_sem1['CURSO'].isin(cursos_primer)]
    conteo = df_fund.groupby('ID')['CURSO'].nunique()
    alumnos_4 = conteo[conteo == len(cursos_primer)].index

    # 2. Alumnos que toman algo válido en S2
    df_sem2_filt = df_sem2[
        (df_sem2['ID'].isin(alumnos_4)) &
        (df_sem2['CURSO'].isin(cursos_permitidos))
    ]
    alumnos_validos = sorted(df_sem2_filt['ID'].unique())

    if not alumnos_validos: return pd.DataFrame(columns=cursos_primer)

    # 3. Pivot y Normalización
    notas = (
        df_sem1[
            (df_sem1['ID'].isin(alumnos_validos)) &
            (df_sem1['CURSO'].isin(cursos_primer))
        ]
        .pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
        .reindex(columns=cursos_primer)
        .fillna(0.0)
    )
    
    # ⚠️ Normalización Estandarizada: (Nota - 4.0) / 3.0 -> [-1, 1] aprox
    # Si usas /7.0 es [0, 1]. Asegúrate de usar la misma que en el Warm Start.
    # En el script maestro anterior usamos (n-4)/3.
    
    # Opción A: (n-4)/3
    def norm(x): return (float(str(x).replace(",", ".")) - 4.0) / 3.0
    notas = notas.applymap(norm)
    
    return notas

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def entrenar_predictor(X_data, Y_data, save_path, max_epochs=500, lr=1e-3, patience=100):
    X_train_val, X_test, Y_train_val, Y_test = train_test_split(
        X_data, Y_data, test_size=0.20, random_state=42
    )
    X_train, X_val, Y_train, Y_val = train_test_split(
        X_train_val, Y_train_val, test_size=0.15, random_state=42
    )

    X_train_t = torch.FloatTensor(X_train); Y_train_t = torch.FloatTensor(Y_train)
    X_val_t   = torch.FloatTensor(X_val);   Y_val_t   = torch.FloatTensor(Y_val)
    X_test_t  = torch.FloatTensor(X_test);  Y_test_t  = torch.FloatTensor(Y_test)

    model = EmbeddingPredictor(X_train.shape[1], Y_train.shape[1]).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val = float('inf'); patience_counter = 0

    for epoch in range(1, max_epochs + 1):
        model.train(); optimizer.zero_grad()
        loss = criterion(model(X_train_t), Y_train_t)
        loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad(): vloss = criterion(model(X_val_t), Y_val_t)

        if vloss.item() < best_val - 1e-9:
            best_val = vloss.item()
            torch.save(model.state_dict(), save_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience: break

    best = EmbeddingPredictor(X_train.shape[1], Y_train.shape[1]).to(DEVICE)
    best.load_state_dict(torch.load(save_path, map_location=DEVICE))
    best.eval()
    with torch.no_grad(): test_mse = criterion(best(X_test_t), Y_test_t).item()
    return test_mse

# =========================
# MAIN
# =========================
def main():
    if not os.path.exists(SAVE_BASE): os.makedirs(SAVE_BASE)
    print(f"📂 Guardando redes en: {SAVE_BASE}")
    print("=== Entrenando Redes Neuronales (3 Cohortes: 2019-2021) ===")

    # 1. Vocabulario TuckER
    entities, relations = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    entity_idxs = {e: i for i, e in enumerate(entities)}
    print(f"   Vocabulario cargado: {len(entities)} entidades.")

    # 2. Cargar Notas (3 Cohortes)
    # 2019
    df_19_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20191.csv"))
    df_19_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20192.csv"))
    # 2020
    df_20_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20201.csv"))
    df_20_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20202.csv"))
    # 2021
    df_21_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20211.csv"))
    df_21_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20212.csv"))

    # 3. Construir Vectores X (Input)
    notas_19 = construir_notas_generacion(df_19_1, df_19_2, CURSOS_PRIMER, CURSOS_PERMITIDOS)
    notas_20 = construir_notas_generacion(df_20_1, df_20_2, CURSOS_PRIMER, CURSOS_PERMITIDOS)
    notas_21 = construir_notas_generacion(df_21_1, df_21_2, CURSOS_PRIMER, CURSOS_PERMITIDOS)

    # Concatenar 3 cohortes
    notas_total = pd.concat([notas_19, notas_20, notas_21], axis=0)
    notas_total = notas_total[~notas_total.index.duplicated(keep="first")]

    print(f"-> Vectores X listos: {len(notas_total)} alumnos (2019+2020+2021).")

    # 4. Entrenar Redes por cada RDIM
    resumen = []
    for rdim in RDIMS:
        # Busca el TuckER entrenado con 3 cohortes
        run_dir   = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim))
        tucker_pt = os.path.join(run_dir, "best_model.pt")
        save_path = os.path.join(SAVE_BASE, f"best_predictor_rdim{rdim}_3cohortes.pt")

        print(f"\n>> rdim={rdim} | {tucker_pt}")
        if not os.path.exists(tucker_pt):
            print("   ⚠️ No existe checkpoint TuckER. Saltando.")
            continue

        try:
            E = cargar_E_weights(tucker_pt) # Embeddings aprendidos por TuckER (Target Y)
            
            # Intersección Alumnos (Notas vs Vocabulario TuckER)
            alumnos_comunes = sorted(set(notas_total.index).intersection(entity_idxs.keys()))
            
            if not alumnos_comunes:
                print("   ⚠️ Sin intersección de alumnos. Saltando.")
                continue

            # Preparar Tensores X, Y
            X_data = np.vstack([notas_total.loc[a].values for a in alumnos_comunes])
            idxs   = [entity_idxs[a] for a in alumnos_comunes]
            Y_data = E[idxs].numpy()

            print(f"   -> Entrenando con {len(alumnos_comunes)} alumnos...")
            test_mse = entrenar_predictor(X_data, Y_data, save_path)
            
            print(f"   ✅ Guardado: {save_path}")
            print(f"   📏 MSE: {test_mse:.6f}")
            resumen.append((rdim, len(alumnos_comunes), test_mse))

        except Exception as e:
            print(f"   ❌ Error: {e}")

    if resumen:
        df_r = pd.DataFrame(resumen, columns=["rdim", "n_alumnos", "mse"])
        out = os.path.join(SAVE_BASE, "resumen_entrenamiento_3cohortes.csv")
        df_r.to_csv(out, index=False)
        print(f"\n✅ Resumen guardado en {out}")

if __name__ == "__main__":
    main()

📂 Guardando redes en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_3cohortes_modelobalanceado
=== Entrenando Redes Neuronales (3 Cohortes: 2019-2021) ===
   Vocabulario cargado: 2368 entidades.
-> Vectores X listos: 2351 alumnos (2019+2020+2021).

>> rdim=1 | C:\Users\56946\TuckER\results\Experimento_warm_start_rdim1_1000epochs_earlystopping_2019_2020_2021_patience400\best_model.pt
   -> Entrenando con 2351 alumnos...
   ✅ Guardado: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_3cohortes_modelobalanceado\best_predictor_rdim1_3cohortes.pt
   📏 MSE: 0.019432

>> rdim=2 | C:\Users\56946\TuckER\results\Experimento_warm_start_rdim2_1000epochs_earlystopping_2019_2020_2021_patience400\best_model.pt
   -> Entrenando con 2351 alumnos...
   ✅ Guardado: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_3cohortes_modelobalanceado\best_predictor_rdim2_3cohortes.pt
   📏 MSE: 0.018852

>> rdim=3 | C:\Users\56946\TuckER\results\Experimento_warm_start_rdim3_100

### probar modelo 

In [10]:
# -*- coding: utf-8 -*-
import os, re, sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from types import SimpleNamespace
from sklearn.metrics import precision_score, recall_score, confusion_matrix

# =====================================
# 1. CONFIGURACIÓN
# =====================================

# Carpeta del Dataset TuckER
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_2021_binario_4d\\"

# Rutas de Datos de Evaluación (Cohorte Test: 2022)
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_INPUT = os.path.join(BASE_PATH, "df_20221.csv")  # Input (S1)
CSV_TARGET = os.path.join(BASE_PATH, "df_20222.csv") # Target (S2)

# Rutas de Modelos
RESULTS_BASE = r"C:\Users\56946\TuckER\results"
PREDICTOR_BASE = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_3cohortes_modelobalanceado"

# Patrones
RUN_PREFIX   = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_2021_patience400"
PREDICTOR_FILE_FMT = "best_predictor_rdim{rdim}_3cohortes.pt"

# Cursos
CURSOS_PRIMER  = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO = ['CC1002','MA1002','MA1102','FI1100']
CURSOS_EVAL    = CURSOS_PRIMER + CURSOS_SEGUNDO

DEVICE = "cpu"
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED)
RDIMS = list(range(1, 17))

# =========================
# UTILIDADES
# =========================
sys.path.append("C:/Users/56946/TuckER")
from load_data import Data

def build_vocab(data_dir, reverse=True):
    if not data_dir.endswith(os.sep): data_dir += os.sep
    d = Data(data_dir=data_dir, reverse=reverse)
    ent2idx = {e:i for i,e in enumerate(d.entities)}
    rel2idx = {r:i for i,r in enumerate(d.relations)}
    return SimpleNamespace(entities=d.entities, relations=d.relations,
                           entity_idxs=ent2idx, relation_idxs=rel2idx)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_tucker_weights(path, device="cpu"):
    sd = pick_state_dict(torch.load(path, map_location=device))
    E = sd["E.weight"].to(device)
    R = sd["R.weight"].to(device)
    W = sd["W"].to(device)
    return E, R, W

def pick_forward_rel_idx(vocab, name):
    if name in vocab.relation_idxs: return vocab.relation_idxs[name]
    cands = [r for r in vocab.relations if r.replace("_reverse","") == name]
    if not cands: cands = [r for r in vocab.relations if re.search(name, r, re.I)]
    if not cands: raise KeyError(f"No encontré relación '{name}' en el vocab.")
    cands.sort(key=lambda r: ("_reverse" in r, r))
    return vocab.relation_idxs[cands[0]]

def contract_M(W, r_vec, d_e):
    if W.shape[0] == r_vec.numel() and W.shape[1] == d_e:
        return torch.tensordot(W, r_vec, dims=([0],[0]))
    if W.shape[1] == r_vec.numel() and W.shape[0] == d_e:
        return torch.tensordot(W, r_vec, dims=([1],[0]))
    raise ValueError(f"Layout W no reconocido: {tuple(W.shape)}")

# ===== Predictor =====
class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),       nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def load_predictor(path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    state = torch.load(path, map_location=device)
    model.load_state_dict(pick_state_dict(state), strict=True)
    model.to(device).eval()
    return model

# ⚠️ FUNCIÓN ROBUSTA PARA NOTAS VACÍAS (INPUT)
def notas_vector(csv_path, alumno_id, cursos_primer):
    df = pd.read_csv(csv_path, sep=';')
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    
    sub = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    
    # Si no tiene nota (NaN o vacía), asignamos 1.0 (nota mínima reprobatoria)
    def clean_n(x):
        try:
            s = str(x).strip()
            if not s or s.lower() == 'nan': return 1.0
            return float(s.replace(",", "."))
        except:
            return 1.0 # Ante la duda, reprueba
            
    sub['NOTA'] = sub['NOTA'].apply(clean_n)

    if sub.empty: 
        # Si no tiene registros, asumimos 1.0 en todo (o ceros, depende de tu lógica de dropout)
        # Aquí usaremos 1.0 normalizado para indicar "le fue mal".
        fake_notes = np.array([1.0]*len(cursos_primer), dtype=np.float32)
        vec = (fake_notes - 4.0) / 3.0
        return torch.tensor(vec).view(1, -1)
    
    series = sub.drop_duplicates(subset=['CURSO']).set_index('CURSO')['NOTA']
    # Rellenamos cursos faltantes con 1.0 (Reprobado)
    series = series.reindex(cursos_primer, fill_value=1.0)
    
    # Normalización (n - 4)/3
    vec = (series.values.astype('float32') - 4.0) / 3.0
    return torch.tensor(vec).view(1, -1)

# =========================
# EVALUACIÓN
# =========================
def evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab):
    print(f"\n=============== Evaluando: {tag} ===============")
    
    # 1. Cargar Modelos
    try:
        E, R, W = load_tucker_weights(tucker_ckpt, device=DEVICE)
        d_e = E.shape[1]
        ridx_apr = pick_forward_rel_idx(vocab, "aprueba")
        try:
            ridx_repr = pick_forward_rel_idx(vocab, "reprueba")
            have_repr = True
        except KeyError:
            have_repr = False # Fallback

        predictor = load_predictor(predictor_ckpt, input_size=len(CURSOS_PRIMER), out_dim=d_e, device=DEVICE)
    except Exception as e:
        print(f"❌ Error cargando modelo: {e}")
        return None

    # 2. Cargar Datos
    df_in = pd.read_csv(CSV_INPUT, sep=';')
    df_out = pd.read_csv(CSV_TARGET, sep=';')
    for df in (df_in, df_out):
        df['ID'] = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    # 3. Filtrar Alumnos Válidos (4 cursos S1)
    ok = (df_in.groupby("ID")["CURSO"].apply(set)
             .apply(lambda s: set(CURSOS_PRIMER).issubset(s)))
    alumnos_validos = ok[ok].index.tolist()

    # 4. Dataframe Evaluación
    df_eval = df_out[(df_out['ID'].isin(alumnos_validos)) &
                     (df_out['CURSO'].isin(CURSOS_EVAL))].copy()

    # 5. Lógica Etiqueta (Target) considerando vacíos = Reprobado
    def clean_target_note(x):
        try:
            s = str(x).strip()
            if not s or s.lower() == 'nan': return 1.0 # Reprobado
            return float(s.replace(",", "."))
        except:
            return 1.0 # Reprobado

    df_eval['VAL_NOTA'] = df_eval['NOTA'].apply(clean_target_note)
    
    # APROB si nota >= 4.0
    df_eval['APROB'] = (df_eval['VAL_NOTA'] >= 4.0).astype(int)
    
    df_eval = df_eval[df_eval['CURSO'].isin(vocab.entity_idxs.keys())].copy()
    
    if df_eval.empty:
        print("⚠️ Sin datos para evaluar.")
        return None

    # 6. Loop Predicción
    ehat_cache = {}
    rows = []
    y_true_cls, y_pred_cls = [], []

    with torch.no_grad():
        M_apr = contract_M(W, R[ridx_apr], d_e)
        M_repr = contract_M(W, R[ridx_repr], d_e) if have_repr else None

        for _, r in df_eval.iterrows():
            aid, curso = r['ID'], r['CURSO']
            
            # Embeddings
            if aid not in ehat_cache:
                x = notas_vector(CSV_INPUT, aid, CURSOS_PRIMER).to(DEVICE)
                ehat_cache[aid] = predictor(x).squeeze(0).cpu()
            
            e_h = ehat_cache[aid]
            e_t = E[vocab.entity_idxs[curso]].cpu()

            # Scores
            s_apr = torch.sigmoid((e_h.view(1,d_e) @ M_apr @ e_t.view(d_e,1)).squeeze()).item()
            if have_repr:
                s_repr = torch.sigmoid((e_h.view(1,d_e) @ M_repr @ e_t.view(d_e,1)).squeeze()).item()
            else:
                s_repr = 1.0 - s_apr

            # Clasificación
            es_reprobado_real = 1 if r['APROB'] == 0 else 0
            predice_reprobado = 1 if s_repr > s_apr else 0
            
            y_true_cls.append(es_reprobado_real)
            y_pred_cls.append(predice_reprobado)

            # Ranking
            if r['APROB'] == 1:
                rank = 1 if s_apr >= s_repr else 2
            else:
                rank = 1 if s_repr >= s_apr else 2

            rows.append({"ID": aid, "CURSO": curso, "hit@1": 1.0 if rank == 1 else 0.0})

    # 7. Métricas
    df_res = pd.DataFrame(rows)
    hits1 = df_res["hit@1"].mean()
    mrr   = (1.0 / (2.0 - df_res["hit@1"])).mean()
    
    prec = precision_score(y_true_cls, y_pred_cls, zero_division=0)
    rec  = recall_score(y_true_cls, y_pred_cls, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true_cls, y_pred_cls).ravel()

    print(f"Precision (Rep): {prec:.4f} | Recall (Rep): {rec:.4f}")
    print(f"Hits@1: {hits1:.4f} | N: {len(df_res)}")

    return {
        "tag": tag,
        "Hits@1": hits1,
        "MRR": mrr,
        "Precision_Rep": prec,
        "Recall_Rep": rec,
        "N": len(df_res)
    }

def main():
    if not os.path.exists(DATA_DIR):
        print(f"❌ Error Data Dir: {DATA_DIR}")
        return

    vocab = build_vocab(DATA_DIR, reverse=True)
    res_list = []
    
    print(f"📂 Evaluando 3 Cohortes en: {PREDICTOR_BASE}")

    for rdim in RDIMS:
        tag = f"rdim{rdim}"
        tucker_ckpt = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim), "best_model.pt")
        predictor_ckpt = os.path.join(PREDICTOR_BASE, PREDICTOR_FILE_FMT.format(rdim=rdim))

        if not os.path.exists(tucker_ckpt) or not os.path.exists(predictor_ckpt):
            continue

        res = evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab)
        if res: res_list.append(res)

    if res_list:
        df_sum = pd.DataFrame(res_list).sort_values("tag")
        out = os.path.join(PREDICTOR_BASE, "resumen_evaluacion_3cohortes_final.csv")
        df_sum.to_csv(out, index=False)
        print("\n=== RESUMEN FINAL ===")
        print(df_sum.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

if __name__ == "__main__":
    main()

📂 Evaluando 3 Cohortes en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_3cohortes_modelobalanceado

=============== Evaluando: rdim1 ===============
Precision (Rep): 0.1324 | Recall (Rep): 0.3869
Hits@1: 0.4897 | N: 3110

=============== Evaluando: rdim2 ===============
Precision (Rep): 0.0937 | Recall (Rep): 0.3472
Hits@1: 0.3502 | N: 3110

=============== Evaluando: rdim3 ===============
Precision (Rep): 0.1758 | Recall (Rep): 0.6845
Hits@1: 0.4286 | N: 3110

=============== Evaluando: rdim4 ===============
Precision (Rep): 0.1358 | Recall (Rep): 0.5675
Hits@1: 0.3447 | N: 3110

=============== Evaluando: rdim5 ===============
Precision (Rep): 0.0937 | Recall (Rep): 0.3472
Hits@1: 0.3502 | N: 3110

=============== Evaluando: rdim6 ===============
Precision (Rep): 0.3407 | Recall (Rep): 0.5476
Hits@1: 0.7550 | N: 3110

=============== Evaluando: rdim7 ===============
Precision (Rep): 0.3407 | Recall (Rep): 0.5476
Hits@1: 0.7550 | N: 3110

=============== Evaluando: rdi